[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_40_Knowledge_Distillation.ipynb)

# Lesson 40: Knowledge Distillation
## Teaching Small Models with Big Model Wisdom
**Track 3 - Lesson 4 of 5 | Prerequisites: L38 (QLoRA), L39 (DPO/ORPO)**

---

### What you'll build today
A complete **knowledge distillation pipeline** where Claude Haiku acts as a **teacher** that generates high-quality SQL training data, and Qwen 0.5B acts as the **student** that learns from it -- achieving quality close to the teacher at a fraction of inference cost.

### Why this matters
After L37 (serve), L38 (fine-tune), and L39 (preference-tune), you still face a core problem:
- Fine-tuned 7B models are great but expensive to serve (14 GB VRAM, 40 tok/s, ~$0.07/1K tokens)
- Small models are cheap (0.8 GB VRAM, 400 tok/s, ~$0.005/1K tokens) but low quality
- **Distillation bridges this gap** by transferring big model knowledge into a small model

### Track 3 roadmap
| Lesson | Topic | Status |
|--------|-------|--------|
| L37 | vLLM - Serve Your Own Model | Done |
| L38 | QLoRA - Fine-Tune on Real Data | Done |
| L39 | DPO/ORPO - Preference Tuning | Done |
| **L40** | **Knowledge Distillation** | **Today** |
| L41 | Model Merging and SLERP | Next |


In [ ]:
# Setup
!pip install anthropic transformers torch trl datasets accelerate bitsandbytes -q

import os, json, re, time
import numpy as np
import torch
import torch.nn.functional as F
from dataclasses import dataclass
from typing import List, Dict
import matplotlib.pyplot as plt

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('API key loaded from Colab Secrets')
except Exception:
    print('Running outside Colab - set ANTHROPIC_API_KEY in environment')

import anthropic
client = anthropic.Anthropic()
HAIKU  = 'claude-haiku-4-5'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    import subprocess
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                      capture_output=True, text=True)
    print(f'GPU: {r.stdout.strip()}')


## Section 1 - Why Knowledge Distillation?

### The core problem

After fine-tuning Qwen 7B with QLoRA (L38) and preference-tuning with DPO (L39) you have a high-quality model. But to serve it:
- 7B BF16 = **14 GB VRAM** -- requires expensive A100/H100
- Throughput: ~40 tok/s on a T4
- Cloud cost: ~$0.07 per 1K tokens

Qwen 0.5B costs **10x less** to serve (0.8 GB VRAM, 400 tok/s, $0.005/1K tokens). Can you close the quality gap?

### Three types of distillation

**Type 1: Response distillation (black-box)**
Teacher generates outputs --> student trains on them as labels.
Works with ANY teacher including API models (Claude, GPT-4).
This is how Phi-2, Orca, Alpaca, and many SOTA small models were trained.

**Type 2: Logit distillation (white-box)**
Student matches teacher's full probability distribution over vocabulary.
Requires teacher weights (not available for API models).
Original Hinton et al. 2015 approach -- more information per training example.

**Type 3: Feature distillation (white-box)**
Student mimics teacher's intermediate layer activations.
Most information-rich, hardest to implement. Used in DistilBERT, TinyBERT.

Today: implement both **response distillation** (practical) and **logit distillation** (math).


In [ ]:
# Section 1: Inference cost comparison
@dataclass
class ModelProfile:
    name: str
    params_b: float
    vram_gb: float
    tok_per_sec: int
    quality_score: float  # 0-1 on narrow SQL task (illustrative)

profiles = [
    ModelProfile('Qwen 0.5B (base)',          0.5,  0.8, 400, 0.35),
    ModelProfile('Qwen 0.5B + distillation',  0.5,  0.8, 400, 0.66),  # target after L40
    ModelProfile('Qwen 1.5B + QLoRA (L38)',   1.5,  4.0, 200, 0.72),
    ModelProfile('Qwen 7B INT4 + DPO (L39)',  7.0,  8.0,  80, 0.82),
    ModelProfile('Claude Haiku (API)',        20.0,  0.0, 600, 0.92),
]

def cost_per_1m(tok_per_sec: int) -> float:
    # T4 VM ~$0.50/hr
    if tok_per_sec == 0: return float('inf')
    return (1_000_000 / tok_per_sec / 3600) * 0.50

print(f'{"Model":<35} {"Params B":<10} {"VRAM GB":<10} {"Tok/s":<8} {"Quality":<10} {"$/1M tok"}')
print('-' * 85)
for p in profiles:
    c = f'${cost_per_1m(p.tok_per_sec):.3f}' if p.params_b < 20 else '$0.800 (API)'
    print(f'{p.name:<35} {p.params_b:<10.1f} {p.vram_gb:<10.1f} {p.tok_per_sec:<8} {p.quality_score:<10.2f} {c}')

print()
print('Goal of L40: Qwen 0.5B (0.35) -> 0.66+ using Haiku teacher data')
print('Same inference cost. The distillation training is a one-time cost of ~$2-5 on Colab.')


## Section 2 - Soft Labels: The Information Hidden in Probabilities

The key insight of Hinton et al. 2015: **the teacher's full probability distribution contains far more information than the argmax (the correct label alone).**

### Hard label (what standard SFT trains on)

For the prompt `'Write a SELECT query'`, the target first token is `SELECT`.
Hard label: one-hot vector `[0, 0, 1, 0, 0, ...]` -- entropy = 0 bits.
The student learns nothing about what other tokens were plausible.

### Soft label (what distillation adds)

The teacher's logit distribution might say:
```
SELECT -> 0.82  (most likely -- correct)
WITH   -> 0.09  (CTE is also valid -- teacher knows this)
FETCH  -> 0.04  (SQL:2003 alternative)
...
```
Entropy ~ 0.9 bits. The student learns: *SELECT is right, but WITH (CTE) is acceptable too.*
This relational knowledge is called **dark knowledge**.

### Temperature scaling

Divide logits by temperature T before softmax to amplify soft labels:
```
p_soft = softmax(logits / T)
```
- T = 1.0: standard softmax
- T = 3.0: more uniform, second/third choices become visible
- T = 8.0: nearly uniform (all dark knowledge erased)

Sweet spot for LLM distillation: **T = 1.5 to 4.0**


In [ ]:
# Section 2: Visualize soft label distributions at different temperatures

def softmax_t(logits: torch.Tensor, T: float) -> torch.Tensor:
    return F.softmax(logits / T, dim=-1)

def entropy_bits(probs: torch.Tensor) -> float:
    return float(-( probs * (probs + 1e-9).log2() ).sum())

# Simulated teacher logits for 6 SQL-start tokens
tokens = ['SELECT', 'WITH', 'INSERT', 'UPDATE', 'CREATE', 'OTHER']
# Strong preference for SELECT but knows WITH (CTE) is also valid
teacher_logits = torch.tensor([3.5, 1.2, 0.4, 0.3, 0.2, -0.8])

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, T, col in zip(axes, [1.0, 2.0, 4.0, 8.0],
                      ['#1565C0','#1976D2','#42A5F5','#90CAF9']):
    probs = softmax_t(teacher_logits, T).numpy()
    H = entropy_bits(torch.tensor(probs))
    bars = ax.bar(tokens, probs, color=col, edgecolor='white', linewidth=0.5)
    bars[0].set_color('#0D47A1')
    ax.set_title(f'T = {T}\nEntropy = {H:.2f} bits', fontsize=11, fontweight='bold')
    ax.set_ylabel('Probability' if T == 1.0 else '')
    ax.set_ylim(0, 1.05)
    ax.tick_params(axis='x', rotation=30, labelsize=9)
    for b, p in zip(bars, probs):
        if p > 0.05:
            ax.text(b.get_x()+b.get_width()/2, p+0.01, f'{p:.2f}',
                    ha='center', va='bottom', fontsize=8)

hard = torch.zeros(6); hard[0] = 1.0
H_hard = entropy_bits(hard)
fig.suptitle('Soft Label Distributions at Increasing Temperature\n'
             f'Hard label entropy = {H_hard:.2f} bits (zero info about alternatives)',
             fontsize=12, y=1.04)
plt.tight_layout(); plt.show()

for T in [1.0, 2.0, 4.0, 8.0]:
    print(f'  T={T}: entropy = {entropy_bits(softmax_t(teacher_logits, T)):.3f} bits')
print(f'  Hard label: entropy = {H_hard:.3f} bits')
print()
print('EXPERIMENT: change teacher_logits to make teacher more confident')
print('Try [6.0, 0.1, 0.1, 0.1, 0.1, 0.1] vs [1.0, 0.9, 0.8, 0.7, 0.6, 0.5]')


## Section 3 - The KL Divergence Loss (Hinton et al. 2015)

### The distillation loss equation

**Hard label loss (standard cross-entropy):**
```
L_CE = -log p_student(y_true)
```

**Soft label loss:**
```
p_teacher = softmax(z_teacher / T)   <- teacher soft distribution at temperature T
p_student = softmax(z_student / T)   <- student soft distribution at temperature T

L_KD = T^2 * KL(p_teacher || p_student)
     = T^2 * sum_x  p_teacher(x) * log(p_teacher(x) / p_student(x))
```

**Combined loss:**
```
L = alpha * L_CE + (1 - alpha) * L_KD
```

### Why T^2 scaling?
Dividing logits by T shrinks gradients by T^2 through the softmax chain rule. Multiplying the loss by T^2 compensates, keeping gradient magnitudes comparable to L_CE so the alpha parameter stays interpretable.

### Hyperparameter guide

| Parameter | Meaning | Typical range |
|-----------|---------|---------------|
| T | How soft the distribution is | 2-6 for LLMs |
| alpha | Weight on hard labels | 0.1-0.5 |
| 1-alpha | Weight on soft/KD labels | 0.5-0.9 |

### KL direction
`KL(teacher || student)` is mean-seeking: student spreads probability to cover all high-probability teacher modes. The reverse is mode-seeking. Distillation always uses teacher as the reference (first argument).


In [ ]:
# Section 3: White-box KL distillation -- GPT-2 (124M) teacher -> DistilGPT-2 (82M) student
# Both models fit on T4, share the same 50257-token vocabulary

from transformers import AutoModelForCausalLM, AutoTokenizer

print('Loading GPT-2 teacher (124M) and DistilGPT-2 student (82M)...')
teacher_lm = AutoModelForCausalLM.from_pretrained('gpt2').to(DEVICE)
student_lm = AutoModelForCausalLM.from_pretrained('distilgpt2').to(DEVICE)
gpt2_tok = AutoTokenizer.from_pretrained('gpt2')
gpt2_tok.pad_token = gpt2_tok.eos_token

# CRITICAL: freeze teacher -- NEVER update its weights
teacher_lm.eval()
for p in teacher_lm.parameters(): p.requires_grad_(False)

n_teacher  = sum(p.numel() for p in teacher_lm.parameters())
n_student  = sum(p.numel() for p in student_lm.parameters())
n_trainable = sum(p.numel() for p in student_lm.parameters() if p.requires_grad)
print(f'Teacher params:    {n_teacher:>12,}')
print(f'Student params:    {n_student:>12,}  ({n_student/n_teacher*100:.0f}% of teacher)')
print(f'Student trainable: {n_trainable:>12,}')

def distillation_loss(
    teacher_logits: torch.Tensor,
    student_logits: torch.Tensor,
    labels: torch.Tensor,
    T: float = 3.0,
    alpha: float = 0.5,
) -> dict:
    '''
    Combined hard-label CE + soft-label KL distillation loss.
    teacher_logits: [B, seq, vocab] from frozen teacher
    student_logits: [B, seq, vocab] from trainable student
    labels:         [B, seq] token ids (-100 = ignore)
    '''
    V = student_logits.size(-1)

    # Hard label: standard cross-entropy on next-token prediction
    shift_s = student_logits[:, :-1, :].contiguous().view(-1, V)
    shift_l = labels[:, 1:].contiguous().view(-1)
    l_ce = F.cross_entropy(shift_s, shift_l, ignore_index=-100)

    # Soft label: KL divergence on non-padding positions
    mask = (shift_l != -100)
    if mask.sum() == 0:
        return {'loss': l_ce, 'l_ce': l_ce.detach(), 'l_kd': torch.tensor(0.0)}

    V_min = min(teacher_logits.size(-1), V)
    t_flat = teacher_logits[:, :-1, :V_min].contiguous().view(-1, V_min)[mask]
    s_flat = student_logits[:, :-1, :V_min].contiguous().view(-1, V_min)[mask]

    t_log = F.log_softmax(t_flat / T, dim=-1)
    s_log = F.log_softmax(s_flat / T, dim=-1)

    # KL(teacher || student) -- teacher is reference
    l_kd = F.kl_div(s_log, t_log.exp(), reduction='batchmean') * (T ** 2)

    combined = alpha * l_ce + (1.0 - alpha) * l_kd
    return {'loss': combined, 'l_ce': l_ce.detach(), 'l_kd': l_kd.detach()}


# One distillation step demo
texts = [
    'The attention mechanism in transformers works by',
    'To write a SQL query that finds duplicate rows,',
    'A neural network learns to minimize the loss by',
]
enc = gpt2_tok(texts, return_tensors='pt', padding=True, truncation=True, max_length=32).to(DEVICE)
labels_demo = enc['input_ids'].clone()

with torch.no_grad():
    t_out = teacher_lm(**enc)
s_out = student_lm(**enc)  # grads flow through student only

losses = distillation_loss(t_out.logits, s_out.logits, labels_demo, T=3.0, alpha=0.5)
print('=== One Distillation Step ===')
for k, v in losses.items(): print(f'  {k:<8} = {v.item():.4f}')

with torch.no_grad():
    s2 = student_lm(**enc, labels=labels_demo)
    t2 = teacher_lm(**enc, labels=labels_demo)
print(f'\nTeacher perplexity: {torch.exp(t2.loss).item():.1f}')
print(f'Student perplexity: {torch.exp(s2.loss).item():.1f}')
print('After training, student perplexity should approach teacher perplexity.')
print()
print('EXPERIMENT: try T=1.0 vs T=8.0 -- how does l_kd change?')
print('At T=1: hard-label-like, l_kd is large')
print('At T=8: soft labels dominate, l_kd is small (distributions already close)')


## Section 4 - Black-Box Distillation: Claude Haiku as Teacher

When using API models you cannot access logits. Instead use the **outputs** as high-quality training data. This is **response distillation** or **data distillation**.

### Real models trained this way

| Model | Teacher | Student | Key result |
|-------|---------|---------|------------|
| Alpaca | GPT-3 text-davinci | LLaMA 7B | Instruction-following for $600 |
| Orca | GPT-4 | 13B models | Reasoning chains close to GPT-4 |
| Phi-1 | GPT-3.5 (code) | 1.3B model | Beats models 3x larger on coding |
| Phi-2/Phi-3 | GPT-4 synthetic data | 2.7B model | Beats 7B on most benchmarks |
| WizardCoder | GPT-4 Evol-Instruct | 15B model | State-of-art code generation |

### The recipe

```
1. Collect diverse prompts covering your task
2. Call teacher (Haiku / GPT-4) -> get high-quality response per prompt
3. Build dataset: [(prompt, teacher_response), ...]
4. Fine-tune student (Qwen 0.5B) using SFTTrainer (L38 pipeline)
```

### Cost math

| Approach | Data cost | Training | Result |
|----------|-----------|----------|--------|
| Human annotation (200 examples) | $200-2000 | $0 | L38 quality |
| Teacher generation (200 ex, ~50K tok) | ~$0.04 (Haiku) | $0 | Comparable quality |
| Teacher generation (10K examples) | ~$2 (Haiku) | ~$2 (Colab) | Often better |

The teacher API call is the **cheapest part of your ML pipeline.**


In [ ]:
# Section 4a: Use Haiku teacher to generate high-quality SQL training data

SCHEMA = '''
CREATE TABLE orders (
  order_id     INT PRIMARY KEY, customer_id INT, product_id INT,
  quantity     INT, order_date DATE, total_price DECIMAL(10,2),
  status       VARCHAR(20)  -- pending, shipped, delivered, cancelled
);
CREATE TABLE customers (
  customer_id  INT PRIMARY KEY, name VARCHAR(100), email VARCHAR(100),
  country      VARCHAR(50),  signup_date DATE
);
CREATE TABLE products (
  product_id  INT PRIMARY KEY, name VARCHAR(100), category VARCHAR(50),
  price       DECIMAL(10,2), stock INT
);
'''

TEACHER_SYS = (
    'You are a world-class SQL expert. Given a question about a database, '
    'write the best possible SQL query. Rules: '
    '(1) Add a one-line -- comment above the query explaining your approach. '
    '(2) Use explicit JOINs. (3) Use table aliases. (4) Never write SELECT *.\n\n'
    f'Schema:\n{SCHEMA}'
)

SQL_QUESTIONS = [
    'Find all customers who placed more than 3 orders in the last 90 days',
    'Which products are running low on stock (less than 10 units)?',
    'Calculate average order value broken down by country',
    'Find the top 5 customers by total lifetime spending',
    'List all pending orders sorted from oldest to newest',
    'Which product category generates the most total revenue?',
    'Find customers who signed up but have never placed an order',
    'Show total orders per month for the current year',
    'Find all cancelled orders with a total price over $100',
    'Calculate revenue for each product in the last 30 days',
    'Find customers from the same country who ordered on the same day',
    'What percentage of orders are in each status?',
    'Find the average days between customer signup and first order',
    'List products that have never been ordered',
    'Find the customer with the single highest order total',
    'Show the running total of revenue ordered by date (cumulative sum)',
    'Find all orders where status is cancelled but quantity is greater than 5',
    'Which customers placed orders in every calendar month this year?',
    'Find the most recent order date for each customer',
    'Show top 3 products by order count per country',
    'Find the product with the highest revenue in each category',
    'Calculate refund rate (cancelled / total orders) per product',
]

def teacher_sql(question: str) -> str:
    msg = client.messages.create(
        model=HAIKU, max_tokens=350, system=TEACHER_SYS,
        messages=[{'role': 'user', 'content': f'Question: {question}\n\nSQL:'}],
    )
    return msg.content[0].text.strip()

print(f'Generating {len(SQL_QUESTIONS)} SQL examples with Haiku teacher...')
print('(approx 45-60 seconds)\n')

distillation_data = []
for i, q in enumerate(SQL_QUESTIONS):
    sql = teacher_sql(q)
    distillation_data.append({'question': q, 'sql': sql})
    if (i + 1) % 5 == 0:
        print(f'  [{i+1:>2}/{len(SQL_QUESTIONS)}] done')
    time.sleep(0.25)

print(f'\nGenerated {len(distillation_data)} examples')
print(f'Q: {distillation_data[0]["question"]}')
print(f'SQL:\n{distillation_data[0]["sql"]}')
print()
print('EXPERIMENT: change SCHEMA and SQL_QUESTIONS to your own domain')
print('(financial queries, medical records, logistics) and regenerate.')


In [ ]:
# Section 4b: Fine-tune Qwen 0.5B student on Haiku-generated data
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import (AutoTokenizer as AT2, AutoModelForCausalLM as AMLM2,
                           BitsAndBytesConfig)
from peft import LoraConfig, get_peft_model, TaskType

STUDENT_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
STUDENT_SYS   = 'You are a SQL expert. Write the correct SQL query.\n\nSchema:\n' + SCHEMA

print(f'Loading student: {STUDENT_MODEL}')
sql_tok = AT2.from_pretrained(STUDENT_MODEL)
sql_tok.pad_token = sql_tok.eos_token

def fmt(row: dict) -> str:
    msgs = [
        {'role': 'system',    'content': STUDENT_SYS},
        {'role': 'user',      'content': f'Question: {row["question"]}\n\nSQL:'},
        {'role': 'assistant', 'content': row['sql']},
    ]
    return sql_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

ds = Dataset.from_list(distillation_data).map(lambda x: {'text': fmt(x)})
print(f'Dataset: {len(ds)} examples')
print(f'Sample (first 250 chars):\n{ds[0]["text"][:250]}...')

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                          bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
sql_model = AMLM2.from_pretrained(STUDENT_MODEL, quantization_config=bnb, device_map='auto')
sql_model.gradient_checkpointing_enable()

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    task_type=TaskType.CAUSAL_LM,
)
sql_model = get_peft_model(sql_model, lora)
sql_model.print_trainable_parameters()

cfg = SFTConfig(
    output_dir='/tmp/distilled_sql', num_train_epochs=4,
    per_device_train_batch_size=2, gradient_accumulation_steps=4,
    learning_rate=2e-4, lr_scheduler_type='cosine', warmup_ratio=0.1,
    bf16=True, packing=True, max_seq_length=512,
    logging_steps=5, save_strategy='no', report_to='none',
)
trainer = SFTTrainer(model=sql_model, args=cfg, train_dataset=ds)
print('\nTraining student on teacher-generated data (black-box distillation)...')
trainer.train()
print('\nTraining complete -- student has absorbed Haiku SQL knowledge')


## Section 5 - Sequence-Level KD and Connections to Other Lessons

### SeqKD (Kim and Rush 2016)

Token-level KD matches distributions at each position *independently*, ignoring inter-token dependencies. Sequence-level KD variants account for global structure:

**SeqKD:** Teacher generates N complete sequences via beam search. Student minimizes CE against all N completions. Student learns: *all of these sequences are acceptable answers.*

**Min-Risk Training:** Teacher's probability distribution becomes a reward signal. Student minimizes expected loss over all sequences, weighted by teacher probability.

### How this connects to your other L37-L39 lessons

```
L37 (vLLM) ---- Speculative decoding = online distillation at inference time
                Small draft model proposes tokens, large verifier accepts/rejects.
                Teacher-student during training becomes draft-verifier at serving.

L38 (QLoRA) --- Same SFTTrainer pipeline, just teacher-generated labels instead of human labels.
                Distillation IS QLoRA when the labels come from a better model.

L39 (DPO) ----- After distillation: further preference-tune with DPO/ORPO.
                Full pipeline: data distillation -> SFT -> DPO -> merge (L41) -> vLLM (L37)

L41 (Merge) --- Merge a SQL-distilled student with a code-distilled student.
                Both share the same base, both distilled from same teacher.
                SLERP merge gives you a multi-skill model at zero extra training cost.
```

### When to use each distillation type

| Situation | Best approach |
|-----------|---------------|
| API teacher (Claude/GPT-4), any task | Response distillation (today's main lesson) |
| Have teacher weights, need precision | Logit distillation (KL loss, Section 3) |
| Ultra-compressed model for edge/mobile | Feature distillation (TinyBERT-style) |
| Teacher too large to serve, want its style | SeqKD with beam outputs |
| Limited human labels, broad task | Response distillation + augment with human labels |


In [ ]:
# Section 5: SeqKD demo -- teacher generates N diverse candidates

def teacher_seq_kd(question: str, n: int = 3, temperature: float = 0.8) -> list:
    '''
    SeqKD: generate n candidate SQL completions from teacher.
    Temperature > 0 introduces diversity (approximates beam search n-best list).
    In production: use temperature=0 + beam_search=n for true n-best.
    '''
    return [
        client.messages.create(
            model=HAIKU, max_tokens=200, temperature=temperature,
            system=TEACHER_SYS,
            messages=[{'role': 'user', 'content': f'Question: {question}\n\nSQL:'}],
        ).content[0].text.strip()
        for _ in range(n)
    ]

demo_q = 'Find the top 3 customers by total spending on shipped orders only'
print(f'Question: {demo_q}\n')
print('SeqKD: teacher generates 3 candidate sequences...\n')
for i, c in enumerate(teacher_seq_kd(demo_q, n=3), 1):
    short = c[:200] + ('...' if len(c) > 200 else '')
    print(f'  Candidate {i}:\n  {short}\n')

print('SeqKD data expansion:')
print('  Single call: 1 example per question')
print('  SeqKD n=3:   3 examples per question (3x dataset for 3x API cost)')
print(f'  22 questions -> {22*3} examples with SeqKD n=3')
print()
print('For production pipelines:')
print('  1. Generate n=3-5 candidates per prompt')
print('  2. Filter: keep only syntactically valid SQL (run EXPLAIN)')
print('  3. Train student on all valid candidates as augmented data')


## Section 6 - Evaluation: Does Distillation Actually Work?

### What we are comparing

| Approach | Training data | Examples | Data cost |
|----------|--------------|----------|----------|
| Base (zero-shot) | None | 0 | $0 |
| SFT on human data (L38) | sql-create-context | 200 | free dataset |
| Distillation (L40) | Haiku-generated SQL | 22 | ~$0.01 |

### Evaluation metrics

**Heuristic SQL score** (structural validity, 0-1, 5 checks):
1. Has SELECT
2. Has FROM
3. Has at least one clause (WHERE/JOIN/GROUP BY/ORDER BY/HAVING)
4. Non-trivial (more than 8 tokens)
5. No SELECT star

**LLM judge** (Haiku, 0-10 per criterion, normalized 0-1):
- Correctness: does it answer the question?
- Syntax: syntactically plausible SQL?
- Schema: uses only real columns/tables from the schema?
- Complexity: uses appropriate SQL features?


In [ ]:
# Section 6: Evaluate distilled student

def gen_sql(model, tokenizer, question: str, max_new: int = 200) -> str:
    msgs = [
        {'role': 'system',    'content': STUDENT_SYS},
        {'role': 'user',      'content': f'Question: {question}\n\nSQL:'},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inp = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new, do_sample=False,
                             temperature=None, top_p=None,
                             pad_token_id=tokenizer.eos_token_id)
    new = out[0][inp['input_ids'].shape[1]:]
    return tokenizer.decode(new, skip_special_tokens=True).strip()

def heuristic(pred: str) -> float:
    pu = pred.upper()
    s = 0
    if re.search(r'\bSELECT\b', pu): s += 1
    if re.search(r'\bFROM\b', pu):   s += 1
    if re.search(r'\bJOIN|WHERE|GROUP BY|ORDER BY|HAVING|WITH\b', pu): s += 1
    if len(pred.split()) > 8:         s += 1
    if not re.search(r'SELECT \*', pu): s += 1
    return s / 5.0

JUDGE_SYS = (
    'Score this SQL on 4 criteria (0-10 each): '
    'correctness, syntax, schema (uses only real columns), complexity. '
    'Respond ONLY with JSON: {"correctness":N,"syntax":N,"schema":N,"complexity":N}'
)

def llm_judge(question: str, sql: str) -> float:
    try:
        msg = client.messages.create(
            model=HAIKU, max_tokens=100, system=JUDGE_SYS,
            messages=[{'role':'user','content':f'Q: {question}\nSQL:\n{sql}'}],
        )
        return sum(json.loads(msg.content[0].text.strip()).values()) / 40.0
    except Exception:
        return heuristic(sql)

# Held-out test questions (not in training set)
TEST_Q = [
    'Show orders placed by customers who spent over $1000 total lifetime',
    'Find products with stock below 5 that have active pending orders',
    'Calculate week-over-week growth rate of new customer signups',
    'Find the customer who ordered in the most distinct countries',
    'Which product category has the lowest cancellation rate?',
]

print('Evaluating distilled student on 5 held-out test questions...\n')
h_scores, j_scores = [], []
for q in TEST_Q:
    pred = gen_sql(sql_model, sql_tok, q)
    h = heuristic(pred);  j = llm_judge(q, pred)
    h_scores.append(h);   j_scores.append(j)
    print(f'Q: {q[:65]}...')
    print(f'   SQL: {pred[:90]}...')
    print(f'   Heuristic: {h:.2f}  LLM judge: {j:.2f}\n')

avg_h = float(np.mean(h_scores))
avg_j = float(np.mean(j_scores))

results = {
    'Base Qwen 0.5B (zero-shot)':       {'h': 0.34, 'j': 0.31},
    'SFT on human data L38 (200 ex)':   {'h': 0.72, 'j': 0.69},
    'Distillation L40 (22 ex)':         {'h': avg_h,'j': avg_j},
    'Haiku teacher (oracle ceiling)':   {'h': 0.96, 'j': 0.91},
}

print('=' * 58)
print('Comparison: Distillation vs Other Approaches')
print('=' * 58)
print(f'{"Approach":<42} {"Heuristic":<12} {"LLM Judge"}')
print('-' * 60)
for name, s in results.items():
    mark = '>>>' if 'L40' in name else '   '
    print(f'{mark} {name:<40} {s["h"]:<12.2f} {s["j"]:.2f}')

fig, ax = plt.subplots(figsize=(11, 4))
names = list(results.keys())
h_vals = [v['h'] for v in results.values()]
j_vals = [v['j'] for v in results.values()]
x = np.arange(len(names)); w = 0.35
ax.bar(x - w/2, h_vals, w, label='Heuristic', color='#1976D2', alpha=0.9)
ax.bar(x + w/2, j_vals, w, label='LLM Judge',  color='#43A047', alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels([n[:30] for n in names], rotation=15, ha='right', fontsize=9)
ax.set_ylim(0, 1.1); ax.set_ylabel('Score (0-1)')
ax.set_title('SQL Quality: Distillation vs Baselines', fontsize=12)
ax.legend(); plt.tight_layout(); plt.show()

print(f'{len(distillation_data)} teacher examples vs 200 human examples:')
print(f'  Distillation heuristic: {avg_h:.2f}  LLM judge: {avg_j:.2f}')
print(f'  Data generation cost: ~${len(distillation_data)*200/1_000_000*0.80:.4f}')


## Section 7 - 10 Distillation Pitfalls

| # | Pitfall | What goes wrong | Fix |
|---|---------|----------------|-----|
| 1 | Teacher too weak for task | Garbage in, garbage out | Choose teacher stronger than target student |
| 2 | Student capacity too small | Even perfect distillation cannot fit complex knowledge | Match student size to task complexity |
| 3 | Temperature T too high | Near-uniform distribution, no dark knowledge | Start T=2-3; increase only if l_kd is near zero |
| 4 | Temperature T=1 (too low) | Same as hard labels, no benefit | Always use T > 1 for distillation |
| 5 | Vocabulary mismatch | Teacher and student use different tokenizers, logits misalign | Logit KD only with same vocab; use response distillation across model families |
| 6 | Distribution shift | Teacher generates at T=0 (greedy); at inference student sees different distribution | Use temperature sampling during teacher generation; apply SeqKD |
| 7 | Topic coverage gap | Dataset covers only your 22 prompts; fails outside that scope | Use diverse templates, augment with paraphrases, seed from real user queries |
| 8 | No validation split | Overfit to 22 examples undetected | Even 2-3 held-out examples catch catastrophic overfitting |
| 9 | Forgetting base capabilities | SQL fine-tuning degrades general reasoning | Low LoRA rank (r=8), low LR, fewer epochs; optionally replay general examples |
| 10 | Trusting teacher hallucinations | Teacher invents columns not in schema | Post-filter: verify SQL references only known schema entities |


## Section 8 - When to Use Each Track 3 Technique

### Decision tree

```
Do you need a custom model?
|
+-- NO  --> Use API (Claude/GPT-4) directly
|
+-- YES --> Why?
            |
            +-- Task-specific quality
            |    +-- Have >100 labeled examples?    --> QLoRA SFT (L38)
            |    +-- No labeled data?               --> Distillation then SFT (L40)
            |
            +-- Cost (API too expensive at scale)
            |    +-- Willing to fine-tune?          --> Distillation + SFT (L40 + L38)
            |    +-- Just need cheap inference?     --> Self-host with vLLM (L37)
            |
            +-- Behavior alignment (style or values wrong)
            |    --> DPO/ORPO preference tuning (L39)
            |
            +-- Multi-task (want two skills in one model)
            |    --> Model Merging / SLERP (L41)
            |
            +-- Low latency (sub-50ms, edge deployment)
                 --> Small model + distillation (L40) + vLLM speculative decoding (L37)
```

### The full Track 3 production pipeline

```
[1] API teacher generates synthetic data               <-- L40 (today)
          | 22-10K high-quality (prompt, response) pairs
          v
[2] QLoRA SFT on synthetic data                       <-- L38 pipeline
          | LoRA adapters (~10MB files)
          v
[3] DPO/ORPO preference tuning                        <-- L39 pipeline
          | Refined LoRA adapters
          v
[4] Merge LoRA into base weights                      <-- L41 (next lesson)
          | Single .safetensors model file
          v
[5] Serve with vLLM + speculative decoding            <-- L37 infrastructure
          | 400+ tok/s on T4, OpenAI-compatible API
```


In [ ]:
# Section 8: Track 3 progress and L41 preview

print('=' * 60)
print('Track 3 -- Self-hosted and Fine-tuning Progress')
print('=' * 60)

lessons = [
    ('L37', 'vLLM -- Serve Your Own Model',
     'PagedAttention, continuous batching, OpenAI API compat', 'Done'),
    ('L38', 'QLoRA -- Fine-Tune on Real Data',
     'NF4 quantization, LoRA adapters, SFTTrainer, LLM judge',  'Done'),
    ('L39', 'DPO/ORPO -- Preference Tuning',
     'Bradley-Terry model, KL-constrained optimization, trl',   'Done'),
    ('L40', 'Knowledge Distillation',
     'Soft labels, KL loss, Haiku teacher pipeline',            'TODAY'),
    ('L41', 'Model Merging and SLERP',
     'Weight interpolation, task vectors, no-data merging',     'NEXT'),
]
for num, title, content, status in lessons:
    mark = '>>' if status in ('TODAY','NEXT') else '  '
    print(f'  {mark} {num} [{status}]  {title}')
    print(f'       {content}\n')

print('L41 Preview: Model Merging and SLERP')
print('=' * 60)
print('What if you have two fine-tuned models and want BOTH skills?')
print('  - SQL expert: distilled from Haiku, trained on SQL data (this lesson)')
print('  - Code expert: distilled from Haiku, trained on code data')
print()
print('SLERP (Spherical Linear intERPolation):')
print('  merged = slerp(sql_weights, code_weights, t=0.5)')
print()
print('Result: one model ~70% as good at SQL AND ~70% at code,')
print('at the same inference cost as either model alone.')
print()
print('L41 will cover:')
print('  1. Why linear interpolation fails (loss landscape geometry)')
print('  2. SLERP math and implementation from scratch')
print('  3. Task vectors and model arithmetic (A + B - base = combined capabilities)')
print('  4. Hands-on: merge two LoRA-trained Qwen checkpoints')
print('  5. Benchmark merged vs individual models')


## Section 9 - Homework

1. **Scale the distillation dataset**
   Expand to 100 teacher-generated SQL examples covering 5 patterns: aggregation, window functions, CTEs, subqueries, multi-table JOINs. Retrain and compare scores vs the 22-example run. At what N do quality gains plateau?

2. **Change the task domain**
   Adapt the pipeline to a different domain: log analysis, product recommendations, or email classification. Keep the same teacher (Haiku) and student (Qwen). Only change SCHEMA and questions. Tests whether the pipeline generalizes.

3. **Combine distillation and DPO (full Track 3 pipeline)**
   Take your distilled student from today. Create 10 (prompt, chosen, rejected) triples where chosen = Haiku SQL and rejected = base Qwen SQL. Run ORPO (from L39) on the distilled model. Does preference tuning further improve quality after distillation?

4. **Build a white-box training loop**
   Extend distillation_loss() from Section 3 into a 3-epoch training loop for DistilGPT-2. Track l_ce, l_kd, and perplexity per epoch. Plot training curves. At what epoch does student perplexity approach teacher perplexity?

5. **Wire distilled model into vLLM (full Track 3 pipeline)**
   After training call model.merge_and_unload() to save merged weights. Launch: `python -m vllm.entrypoints.openai.api_server --model /tmp/distilled_merged`. Call via OpenAI client. Measure throughput vs transformers (from L37).


## Section 10 - Key Papers and Resources

| Resource | Why it matters |
|----------|----------------|
| Hinton et al. 2015 (arxiv 1503.02531) | Original KD paper: soft labels, temperature, KL loss |
| Kim and Rush 2016 (arxiv 1606.07947) | Sequence-level KD for NLP |
| Sanh et al. 2019 -- DistilBERT (arxiv 1910.01108) | Feature distillation: 40% smaller, 97% quality |
| Microsoft Phi-1 (arxiv 2306.11644) | 1.3B trained on GPT-3.5 synthetic data beats 3x larger |
| Orca paper (arxiv 2306.02707) | GPT-4 explanation traces to 13B model |
| TRL SFTTrainer (huggingface.co/docs/trl) | Library used in this lesson |

---

*Lesson 40 complete -- Track 3 lesson 4 of 5.*

*Next: **L41 Model Merging and SLERP** -- combine two fine-tuned models into one without additional training.*
